# Cellpose：从分割标签到颗粒面积与周长

本 Notebook 读取二维灰度图或 RGB 图，获得每个颗粒的像素标签，计算面积和周长，剔除接触视野边缘的颗粒，并保存结果。

> **Mask（掩膜）**：表示图像中目标区域的数组。这里采用**实例标签图（Instance Label Map）**：0 表示背景，每个正整数表示一个颗粒。同一颗粒的像素具有相同编号。

先按 [安装说明](README.md) 准备环境，并选择相应 Python 内核。默认使用几何参考标签；将 `SOURCE` 改为 `"cellpose"` 可运行模型预测。

模型与输入输出的完整说明见 [模型与输入输出](README.md#模型与输入输出)。

Cellpose-SAM 是实例分割模型。本节输入灰度电镜图或三通道图像，输出与分析区域大小相同的整数标签图。背景编号为 0，每个独立目标占用一个正整数编号。模型根据图像特征判断每个像素属于哪个实例。

| 数据 | 本节含义 |
| --- | --- |
| 输入图像 | `(H, W)` 灰度图或 `(H, W, 3)` 三通道图；灰度图复制为三个通道传入模型 |
| `masks` | `(H, W)` 实例标签图，供颗粒计数和几何测量使用 |
| `flows` | 模型返回的辅助信息（本节不使用） |
| `styles` | 为接口兼容保留的辅助返回项 |

例如，标签 1 覆盖 100 个像素，则该颗粒的面积为 100 平方像素；后续代码沿其区域边界估计周长，并根据边界上的编号定位接触边缘的目标。


In [ ]:
from pathlib import Path
import json
import hashlib
import importlib.metadata
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tifffile
from skimage import io, color, segmentation, measure
from IPython.display import display

# 支持从 cellpose 目录、第二课目录、课程仓库根目录或其父目录启动。
candidates = [Path.cwd(), Path.cwd() / '4cellpose',
              Path.cwd() / 'lesson02/4cellpose',
              Path.cwd() / 'ai-for-mat-science/lesson02/4cellpose']
ROOT = next((p for p in candidates if (p / '素材/09_TiO2_SEM_1908248.tif').exists()), None)
if ROOT is None:
    raise FileNotFoundError('请在 VS Code 中打开 lesson02/4cellpose 文件夹。')
ASSETS = ROOT / '素材'
FIXTURES = ROOT / 'scripts/fixtures'
OUT = ROOT / 'outputs'
OUT.mkdir(exist_ok=True)
print('材料目录：', ROOT.resolve())


## 1. 选择图像和标签来源

| `SOURCE` | 运行方式 |
| --- | --- |
| `"reference"` | 模拟图 + 已知几何标签，验证测量流程 |
| `"cellpose"` | 运行 Cellpose，为所选图像生成预测标签 |
| `"saved"` | 读取与原图对应的已保存整数 TIFF 标签 |

> **TIFF（Tagged Image File Format，标签图像文件格式）**：用于保存图像的格式；本节以整数像素值保存颗粒编号。

`EDGE_MARGIN_PX = 0` 表示仅剔除实际接触四边的颗粒；设置为 5 时，额外剔除进入边界内侧 5 像素区域的颗粒。

`PIXEL_SIZE_UM` 表示每像素的微米数。默认 `None` 使用像素单位。填写前，根据显微图标尺或仪器元数据完成标定。本节换算适用于横纵尺寸相同的正方形像素。

可选案例（原始文件见 `素材`）：

| IMAGE_NAME | 内容与观察重点 |
| --- | --- |
| `09_TiO2_SEM_1908248.tif` | TiO₂ 扫描电镜颗粒，测量面积和周长 |
| `10_TiO2_SEM_1908250.tif` | 第二个电镜视野，观察颗粒分布和粘连 |
| `11_TiO2_TSEM_1908249.tif` | 透射扫描电镜，观察成像方式与对比度 |
> **SEM（Scanning Electron Microscopy，扫描电子显微术）**：通过扫描电子束获得样品表面相关信息的成像方法。**TSEM（Transmission Scanning Electron Microscopy，透射扫描电子显微术）**：在扫描过程中利用透射电子形成图像的方法。

案例来源见 [素材说明](素材/README.md)。本节默认以像素计量；物理尺寸换算需使用各原始图对应的标定值。

> **ROI（Region of Interest，感兴趣区域）**：从原图中选取的分析范围。本节电镜图选取前 712 行颗粒区域，原图底部仪器信息和标尺保留在原文件中。边缘剔除按 ROI 边界执行，质心坐标以 ROI 左上角为原点。

第一张 TiO₂ SEM 原图信息栏标注视野宽度为 7.520 μm，横向为 1024 像素。分析这张图时可设置 `PIXEL_SIZE_UM = 7.520 / 1024`，计算得到 μm² 面积和 μm 周长。


In [ ]:
SOURCE = 'reference'
IMAGE_NAME = '09_TiO2_SEM_1908248.tif'  # cellpose / saved 模式使用
# BAM 电镜原图底部 56 行为仪器信息栏，分析时取前 712 行。
ROI = (0, 712, 0, 1024) if 'TiO2_' in IMAGE_NAME else None  # y0, y1, x0, x1
SAVED_MASK = OUT / f'{Path(IMAGE_NAME).stem}_cellpose_mask_labels.png'
MODEL_NAME = 'cpsam'
DEVICE = 'cpu'  # 'cpu'、'cuda' 或 'mps'
PIXEL_SIZE_UM = None
EDGE_MARGIN_PX = 0
DIAMETER_PX = None  # 可按图像中典型目标直径填写像素值
FLOW_THRESHOLD = 0.4
CELLPROB_THRESHOLD = 0.0
MIN_SIZE_PX = 15

if SOURCE not in {'reference', 'cellpose', 'saved'}:
    raise ValueError('SOURCE 应为 reference、cellpose 或 saved。')
image_path = (FIXTURES / '03_particles.png') if SOURCE == 'reference' else (ASSETS / IMAGE_NAME)
image = io.imread(image_path)
active_roi = ROI if SOURCE != 'reference' else None
if active_roi is not None:
    y0, y1, x0, x1 = active_roi
    image = image[y0:y1, x0:x1]
if not (image.ndim == 2 or (image.ndim == 3 and image.shape[-1] == 3)):
    raise ValueError('请为本节选择二维灰度图或最后一维为 3 通道的 RGB 图。')
image_gray = color.rgb2gray(image) if image.ndim == 3 else image
image_shape = image.shape[:2]
if not np.isfinite(image).all():
    raise ValueError('请提供像素值有限的图像。')
fig, ax = plt.subplots(figsize=(6, 4))
ax.imshow(image, cmap='gray')
ax.set(title=image_path.name, xlabel='x / pixel', ylabel='y / pixel')
plt.tight_layout()
plt.show()
print('图像形状：', image.shape)


## 2. 读取标签或运行模型

> **CPU（Central Processing Unit，中央处理器）**：执行程序指令的处理器。**GPU（Graphics Processing Unit，图形处理器）**：通过并行计算加速神经网络运算的处理器。
>
> **SAM（Segment Anything Model，通用分割模型）**：图像分割基础模型。这里的 `cpsam` 权重对应 Cellpose-SAM。

Cellpose 返回 `masks`、`flows` 和 `styles`。`masks` 是本节测量使用的整数标签图；`flows` 和 `styles` 是接口返回的辅助项，本节不使用。每个标签的数值用于识别目标，编号大小与颗粒尺寸分别由不同数据描述。

灰度图在代码中复制为三个通道后传入模型，RGB 图保留三个原始通道。

> **RGB（Red, Green, Blue，红绿蓝）**：使用红、绿、蓝三个通道表示图像像素的方式。

`channel_axis=-1` 指定最后一维为通道。首次加载模型会从官方服务器获取权重，下载步骤见 README。

`diameter` 描述典型颗粒直径；`cellprob_threshold` 控制纳入目标区域的像素范围；`min_size` 控制保留实例的最小像素数。调整后，通过原图叠加轮廓核对结果。


In [ ]:
def save_mask_png(mask, path):
    """把整数标签图保存为 16 位 PNG。"""
    mask = np.asarray(mask)
    if mask.size and int(mask.max()) > np.iinfo(np.uint16).max:
        raise ValueError('标签编号超过 PNG 上限 65535，请改用 TIFF。')
    io.imsave(path, mask.astype(np.uint16), check_contrast=False)

def save_mask_color(mask, path):
    """把标签图保存为彩色 PNG，便于直接查看。"""
    rgb = color.label2rgb(np.asarray(mask), bg_label=0, bg_color=(0, 0, 0))
    io.imsave(path, (rgb * 255).astype(np.uint8), check_contrast=False)

def save_mask_json(mask, path):
    """按对象保存标签图：每个颗粒一条记录，points_xy 为该对象的像素坐标。"""
    mask = np.asarray(mask)
    objects = []
    for region in measure.regionprops(mask):
        y0, x0, y1, x1 = region.bbox
        objects.append({
            'label': int(region.label),
            'area_px2': int(region.area),
            'bbox_y0_x0_y1_x1': [int(y0), int(x0), int(y1), int(x1)],
            'points_xy': [[int(x), int(y)] for y, x in region.coords],
        })
    record = {
        'shape': list(mask.shape),
        'background': 0,
        'object_count': len(objects),
        'encoding': '每个对象一条记录；points_xy 为该对象的像素坐标 [x, y]',
        'objects': objects,
    }
    path.write_text(json.dumps(record, ensure_ascii=False), encoding='utf-8')

if SOURCE == 'reference':
    source_mask_path = FIXTURES / '03_particles_reference_mask.tif'
    masks = tifffile.imread(source_mask_path)
    provenance = '课程生成的几何参考标签'
elif SOURCE == 'saved':
    source_mask_path = Path(SAVED_MASK)
    masks = io.imread(source_mask_path)
    provenance = '从文件读取的标签；来源见原预测记录'
else:
    import torch
    from cellpose import models
    if DEVICE == 'cuda' and not torch.cuda.is_available():
        raise RuntimeError('请安装适配显卡的 PyTorch，或将 DEVICE 设置为 cpu。')
    if DEVICE == 'mps' and not torch.backends.mps.is_available():
        raise RuntimeError('请使用支持 MPS 的环境，或将 DEVICE 设置为 cpu。')
    model_key = (MODEL_NAME, DEVICE)
    if globals().get('_loaded_model_key') != model_key:
        model = models.CellposeModel(
            pretrained_model=MODEL_NAME, device=torch.device(DEVICE),
            gpu=DEVICE != 'cpu', use_bfloat16=False)
        _loaded_model_key = model_key
    torch.set_num_threads(4)  # 小图 CPU 推理使用 4 个线程
    image_3ch = np.repeat(image[..., None], 3, axis=-1) if image.ndim == 2 else image
    masks, flows, styles = model.eval(
        image_3ch, channel_axis=-1, diameter=DIAMETER_PX,
        flow_threshold=FLOW_THRESHOLD, cellprob_threshold=CELLPROB_THRESHOLD,
        min_size=MIN_SIZE_PX, batch_size=1)
    source_mask_path = OUT / f'{image_path.stem}_cellpose_mask_labels.png'
    save_mask_png(masks, source_mask_path)
    provenance = f'Cellpose {importlib.metadata.version("cellpose")} / {MODEL_NAME} 预测'
print('标签来源：', provenance)
print('标签文件：', source_mask_path)
print('彩色标签图将保存为：', OUT / f'{image_path.stem}_{SOURCE}_mask.png')


## 3. 解读一个颗粒的标签

`masks[y, x]` 给出第 y 行、第 x 列像素所属的目标。`masks == label_id` 得到某一个颗粒的二值区域。区域中 True 的数量就是颗粒面积，单位为平方像素。

> **二值掩膜（Binary Mask）**：仅用 True 和 False 表示目标内部与外部的数组。

颗粒数量取决于实际出现的正整数编号。参考图特意使用跳跃编号，方便观察编号与数量之间的关系。


In [ ]:
if masks.shape != image_shape or masks.ndim != 2:
    raise ValueError('标签图应与原图具有相同的二维形状。')
if not np.issubdtype(masks.dtype, np.integer) or (masks < 0).any():
    raise ValueError('标签应为非负整数，背景值为 0。')
ids = np.unique(masks)
ids = ids[ids > 0]
print('颗粒编号：', ids.tolist())
print('颗粒数量：', len(ids))
print('标签图左上角 8×12 像素：', masks[:8, :12], sep=chr(10))
if len(ids):
    label_id = int(ids[0])
    particle = masks == label_id
    y, x = np.argwhere(particle)[len(np.argwhere(particle)) // 2]
    print(f'像素 masks[{y}, {x}] = {masks[y, x]}')
    print(f'编号 {label_id} 的面积：{particle.sum()} px²')
    fig, axes = plt.subplots(1, 2, figsize=(9, 4))
    axes[0].imshow(color.label2rgb(masks, image=image, bg_label=0, alpha=0.4))
    axes[0].set_title('Instance labels')
    axes[1].imshow(particle, cmap='gray')
    axes[1].set_title(f'Label {label_id}: binary mask')
    for ax in axes:
        ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('当前标签包含 0 个颗粒；后续将输出空表。')


## 4. 计算面积、周长并标记边缘颗粒

面积 A 等于标签区域的像素数。周长 P 采用四方向 Crofton 估计，由边界与不同方向直线的交点数估计长度。

> **Crofton 周长（Crofton Perimeter）**：一种基于积分几何的数字图像周长估计方法。轮廓的像素化、分辨率和分割误差会影响估计值；区域内部孔洞的边界也计入本节周长。

函数先收集四条边上的标签，再将对应颗粒整体标记为 `touches_edge=True`。保留颗粒的编号与原标签保持一致。

对于像素尺寸 s（μm/像素），面积为 A×s²（μm²），周长为 P×s（μm）。颗粒面积表示当前图像中的二维投影面积。


In [ ]:
def measure_particles(labels, pixel_size_um=None, edge_margin_px=0):
    """测量二维整数标签，返回全部对象表与剔除边缘后的标签。"""
    labels = np.asarray(labels)
    if labels.ndim != 2 or min(labels.shape) == 0:
        raise ValueError('labels 应为非空二维数组。')
    if not np.issubdtype(labels.dtype, np.integer) or (labels < 0).any():
        raise ValueError('labels 应包含非负整数。')
    if not isinstance(edge_margin_px, (int, np.integer)) or edge_margin_px < 0:
        raise ValueError('边缘宽度应为非负整数。')
    if pixel_size_um is not None and (not np.isfinite(pixel_size_um) or pixel_size_um <= 0):
        raise ValueError('像素尺寸应为有限正数。')
    # margin=0 检查最外侧一行/列；margin=5 检查距边界 <=5 的像素。
    width = edge_margin_px + 1
    edge_ids = np.unique(np.concatenate([
        labels[:width, :].ravel(), labels[-width:, :].ravel(),
        labels[:, :width].ravel(), labels[:, -width:].ravel()]))
    edge_ids = edge_ids[edge_ids > 0]
    rows = []
    for region in measure.regionprops(labels):
        label_id = int(region.label)
        # 周围补背景，确保局部切片中的外轮廓完整参与周长估计。
        local = np.pad(region.image, 1)
        rows.append({
            'label': label_id,
            'area_px2': int(region.area),
            'perimeter_px': float(measure.perimeter_crofton(local, directions=4)),
            'centroid_y_px': float(region.centroid[0]),
            'centroid_x_px': float(region.centroid[1]),
            'touches_edge': bool(label_id in edge_ids),
            'kept': bool(label_id not in edge_ids),
        })
    columns = ['label', 'area_px2', 'perimeter_px', 'centroid_y_px',
               'centroid_x_px', 'touches_edge', 'kept']
    table = pd.DataFrame(rows, columns=columns).astype({
        'label': 'int64', 'area_px2': 'int64', 'perimeter_px': 'float64',
        'centroid_y_px': 'float64', 'centroid_x_px': 'float64',
        'touches_edge': 'bool', 'kept': 'bool'})
    if pixel_size_um is not None:
        table['area_um2'] = table['area_px2'] * pixel_size_um**2
        table['perimeter_um'] = table['perimeter_px'] * pixel_size_um
    filtered = labels.copy()
    filtered[np.isin(labels, edge_ids)] = 0
    return table, filtered

all_particles, filtered_masks = measure_particles(masks, PIXEL_SIZE_UM, EDGE_MARGIN_PX)
kept_particles = all_particles.loc[all_particles['kept']].copy()
print(f'全部：{len(all_particles)}；边缘剔除：{all_particles.touches_edge.sum()}；保留：{len(kept_particles)}')
display(all_particles.round(3))


## 5. 核对过滤前后的轮廓

中图用绿色标出保留的颗粒，用红色标出边缘颗粒。右图显示过滤后的标签。颜色用于辅助区分区域，表格中的原始编号用于追踪每个颗粒。


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(image, cmap='gray')
axes[0].set_title('Original')
display_image = image.astype(float) / max(float(image.max()), 1.0)
rgb = color.gray2rgb(display_image) if image.ndim == 2 else display_image.copy()
kept_boundary = segmentation.find_boundaries(filtered_masks, mode='inner')
removed_masks = np.where(np.isin(masks, all_particles.loc[all_particles.touches_edge, 'label']), masks, 0)
removed_boundary = segmentation.find_boundaries(removed_masks, mode='inner')
rgb[kept_boundary] = [0.1, 1.0, 0.1]
rgb[removed_boundary] = [1.0, 0.1, 0.1]
axes[1].imshow(rgb)
axes[1].set_title('Green: kept / Red: edge')
for row in all_particles.itertuples():
    axes[1].text(row.centroid_x_px, row.centroid_y_px, str(row.label),
                 ha='center', va='center', color='yellow', fontsize=8)
axes[2].imshow(color.label2rgb(filtered_masks, image=image, bg_label=0, alpha=0.45))
axes[2].set_title('Complete particles')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
stem = f'{image_path.stem}_{SOURCE}'
fig.savefig(OUT / f'{stem}_overlay.png', dpi=160, bbox_inches='tight')
plt.show()


## 6. 查看分布并保存结果

统计表同时保留所有颗粒的测量值和筛选状态。尺寸分析使用 `kept_particles` 中的完整颗粒。边缘对象的测量值描述当前视野内可见的区域。

> **CSV（Comma-Separated Values，逗号分隔值）**：按行和列保存数据的文本格式。本节使用 UTF-8 编码并附加字节顺序标记，方便 Excel 识别。
>
> **UTF-8（Unicode Transformation Format — 8-bit，8 位 Unicode 转换格式）**：使用变长字节序列存储字符的编码方式。**BOM（Byte Order Mark，字节顺序标记）**：位于文本开头的编码识别标记。

颗粒识别结果决定后续尺寸统计的质量。合并的颗粒、漏检的颗粒和偏移的边界，都可以通过叠加图与原图比较进行核查。边缘筛选后得到的是当前视野内完整颗粒的分布；大颗粒接触边缘的概率较高，研究中可采用多个视野与统一采样规则。


In [ ]:
area_col = 'area_um2' if PIXEL_SIZE_UM is not None else 'area_px2'
perimeter_col = 'perimeter_um' if PIXEL_SIZE_UM is not None else 'perimeter_px'
if len(kept_particles):
    display(kept_particles[[area_col, perimeter_col]].describe().round(3))
    fig, axes = plt.subplots(1, 2, figsize=(9, 3))
    for ax, column in zip(axes, [area_col, perimeter_col]):
        ax.hist(kept_particles[column], bins=min(10, len(kept_particles)), edgecolor='white')
        ax.set(xlabel=column, ylabel='Count')
    plt.tight_layout()
    plt.show()

all_particles.to_csv(OUT / f'{stem}_all_particles.csv', index=False, encoding='utf-8-sig')
kept_particles.to_csv(OUT / f'{stem}_kept_particles.csv', index=False, encoding='utf-8-sig')
mask_labels_path = OUT / f'{stem}_mask_labels.png'
save_mask_png(masks, mask_labels_path)
mask_path = OUT / f'{stem}_mask.png'
save_mask_color(masks, mask_path)
mask_json_path = OUT / f'{stem}_mask.json'
save_mask_json(masks, mask_json_path)
filtered_mask_path = OUT / f'{stem}_filtered_mask.png'
save_mask_color(filtered_masks, filtered_mask_path)
filtered_mask_labels = OUT / f'{stem}_filtered_mask_labels.png'
save_mask_png(filtered_masks, filtered_mask_labels)
filtered_mask_json = OUT / f'{stem}_filtered_mask.json'
save_mask_json(filtered_masks, filtered_mask_json)
def version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return 'not installed'
record = {
    'source': SOURCE, 'provenance': provenance,
    'image': str(image_path), 'mask': str(mask_path),
    'mask_labels': str(mask_labels_path), 'mask_json': str(mask_json_path),
    'filtered_mask': str(filtered_mask_path),
    'filtered_mask_labels': str(filtered_mask_labels),
    'filtered_mask_json': str(filtered_mask_json),
    'image_sha256': hashlib.sha256(image_path.read_bytes()).hexdigest(),
    'mask_sha256': hashlib.sha256(mask_labels_path.read_bytes()).hexdigest(),
    'image_shape': list(image.shape), 'roi_y0_y1_x0_x1': active_roi,
    'pixel_size_um': PIXEL_SIZE_UM, 'edge_margin_px': EDGE_MARGIN_PX,
    'perimeter_method': 'Crofton, 4 directions; includes hole boundaries',
    'prediction_parameters': ({'model': MODEL_NAME, 'device': DEVICE,
        'diameter_px': DIAMETER_PX, 'flow_threshold': FLOW_THRESHOLD,
        'cellprob_threshold': CELLPROB_THRESHOLD, 'min_size_px': MIN_SIZE_PX,
        'use_bfloat16': False, 'batch_size': 1, 'channel_axis': -1}
        if SOURCE == 'cellpose' else None),
    'counts': {'all': len(all_particles), 'kept': len(kept_particles),
               'removed': int(all_particles.touches_edge.sum())},
    'versions': {name: version(name) for name in
                 ['cellpose', 'torch', 'numpy', 'pandas', 'scikit-image', 'tifffile']},
    'python': sys.version,
}
(OUT / f'{stem}_metadata.json').write_text(json.dumps(record, ensure_ascii=False, indent=2), encoding='utf-8')
print('结果保存位置：', OUT.resolve())


## 7. 用已知几何检查测量步骤

参考图中有 5 个内部颗粒和 4 个边缘颗粒。编号 1 是半径 20 像素的圆。其连续几何面积为 πr²，周长为 2πr；像素标签的测量值会随数字化产生小幅差异。

下面的检查验证参考图计数、四边剔除、原编号保留以及单位换算。它检验测量代码；模型预测质量通过独立标注与预测结果比较评估。


In [ ]:
reference = tifffile.imread(FIXTURES / '03_particles_reference_mask.tif')
reference_table, reference_filtered = measure_particles(reference)
assert len(reference_table) == 9
assert reference_table.kept.sum() == 5
assert set(reference_table.loc[reference_table.touches_edge, 'label']) == {21, 25, 30, 42}
assert set(np.unique(reference_filtered)) == {0, 1, 3, 7, 12, 16}
assert np.all(reference_filtered[reference_filtered > 0] == reference[reference_filtered > 0])
calibrated, _ = measure_particles(reference, pixel_size_um=0.1)
np.testing.assert_allclose(calibrated.area_um2, reference_table.area_px2 * 0.01)
np.testing.assert_allclose(calibrated.perimeter_um, reference_table.perimeter_px * 0.1)
circle = reference_table.set_index('label').loc[1]
display(pd.DataFrame({
    'quantity': ['area_px2', 'perimeter_px'],
    'continuous_circle': [np.pi * 20**2, 2 * np.pi * 20],
    'pixel_measurement': [circle.area_px2, circle.perimeter_px]
}).round(3))
print('几何参考案例检查通过。')


参考：[Cellpose 模型](https://cellpose.readthedocs.io/en/latest/models.html)、[Cellpose 接口](https://cellpose.readthedocs.io/en/latest/api.html)、[Crofton 周长与区域测量](https://scikit-image.org/docs/stable/api/skimage.measure.html)、[案例图片来源](素材/README.md)。
